# Explainability — Grad-CAM + **physics-informed** Perona-Malik maps

This notebook explains **federated** ResNet18 predictions on **all three** medical datasets
(Brain Tumor, Colon histopathology, COVID-19) unless you narrow the list in the config cell.

## What is (and is not) "physics-based" here

| Method | Role |
|--------|------|
| **Grad-CAM** (`layer4`) | Standard **gradient-based** attribution: where the **classifier** looked. This is **not** a physics PDE — it is backprop signal on CNN features. |
| **Perona–Malik grid map** (`feature_layer`, same as PIDL training) | **Physics-informed**: each grid cell scores the squared **Perona–Malik anisotropic diffusion residual** on the feature map — the same spatial operator used in your PIDL regularizer. High values mark regions with strong **edge/texture structure** under that operator. |
| **Overlap (IoU @ top 25%)** | Checks whether **classifier attention** (Grad-CAM) aligns with **PIDL-style spatial structure** (PM map). |

## Checkpoints (`final_model.pth`)

This notebook **always runs a full Flower FL training** per dataset under
`results_explainability/{dataset}/{n}_clients/` (it does **not** copy weights from notebook 01).
After each run, `finalize_experiment()` and `save_final_model_from_session()` ensure the
global `state_dict` is written to disk before explainability plots.

The download step zips the **entire** `results_explainability/` folder (figures, CSVs, **`.pth`** files).


---
## A. Setup


In [2]:
# Optional: Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

import os, sys, json, gc, time
from pathlib import Path

GITHUB_REPO = 'https://github.com/PulockDas/medical_fl_pidl.git'
PROJECT_ROOT = Path('/content/medical_fl_pidl')

if not PROJECT_ROOT.exists():
    os.system(f'git clone {GITHUB_REPO} {PROJECT_ROOT}')
    if not PROJECT_ROOT.exists():
        raise RuntimeError('Clone failed — set GITHUB_REPO.')
else:
    os.system(f'git -C {PROJECT_ROOT} pull')

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

!pip install -q --upgrade pip setuptools wheel
!pip install -q -r requirements.txt

import torch
print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 46.0.7 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
pyopenssl 24.2.1 requires cryptography<44,>=41.0.5,

---
## B. Global configuration — **all datasets by default**


In [3]:
from __future__ import annotations

import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

KAGGLE_SLUGS = {
    'brain_tumor_mri':          'masoudnickparvar/brain-tumor-mri-dataset',
    'colon_cancer_or_pathology': 'andrewmvd/lung-and-colon-cancer-histopathological-images',
    'covid':                    'tawsifurrahman/covid19-radiography-database',
}
COLON_OR_LUNG = 'colon_image_sets'

# Default: explain all three datasets (edit to subset e.g. ['brain_tumor_mri'])
DATASETS_TO_RUN = [
    'brain_tumor_mri',
    'colon_cancer_or_pathology',
    'covid',
]

num_clients     = 3
num_rounds      = 5
local_epochs    = 2
batch_size      = 32
learning_rate   = 1e-3
image_size      = 224
random_seed     = 42
feature_layer   = 'layer2'
grid_size       = 4
lambda_pm       = 0.10
regularizer_type = 'perona_malik'
use_grid_loss   = True
k_pm            = 1.0
use_secagg      = False

samples_per_class = 5
gradcam_layer     = 'layer4'

EXPLAIN_ROOT = PROJECT_ROOT / 'results_explainability'
EXPLAIN_ROOT.mkdir(parents=True, exist_ok=True)

print('Datasets:', DATASETS_TO_RUN)
print('Output root:', EXPLAIN_ROOT)


Datasets: ['brain_tumor_mri', 'colon_cancer_or_pathology', 'covid']
Output root: /content/medical_fl_pidl/results_explainability


---
## C. Helpers: data root, weights, one full explainability pass


In [4]:
import kagglehub
from data.kaggle_loader import find_image_root


def download_data_root(dataset_name: str) -> str:
    slug = KAGGLE_SLUGS[dataset_name]
    raw = kagglehub.dataset_download(slug)
    print(f'  [{dataset_name}] downloaded -> {raw}')
    if dataset_name == 'colon_cancer_or_pathology':
        cand = list(Path(raw).rglob(COLON_OR_LUNG))
        return str(cand[0]) if cand else str(find_image_root(raw))
    return str(find_image_root(raw))


def _pth_ok(p: Path) -> bool:
    return p.is_file() and p.stat().st_size > 1024


def ensure_final_model(dataset_name: str, data_root: str, result_dir: Path) -> Path:
    """Always re-train FL in this notebook; write final_model.pth under result_dir."""
    result_dir.mkdir(parents=True, exist_ok=True)
    model_path = result_dir / 'final_model.pth'
    if model_path.exists():
        model_path.unlink(missing_ok=True)
        print(f'  Removed stale {model_path.name} — training fresh FL run.')

    print('  Running Flower FL (required for this notebook — no reuse of notebook 01 weights)...')
    import json as _json
    from flwr.simulation import run_simulation
    from federated.server_app import app as _server_app
    from federated.client_app import _make_client_app

    run_cfg = {
        'dataset_name': dataset_name,
        'data_root': data_root,
        'log_dir': str(result_dir),
        'num_classes': 0,
        'num_clients': num_clients,
        'num_server_rounds': num_rounds,
        'min_fit_clients': num_clients,
        'local_epochs': local_epochs,
        'batch_size': batch_size,
        'learning_rate': learning_rate,
        'image_size': image_size,
        'feature_layer': feature_layer,
        'regularizer_type': regularizer_type,
        'lambda_pm': lambda_pm,
        'use_grid_loss': use_grid_loss,
        'grid_size': grid_size,
        'k': k_pm,
        'random_seed': random_seed,
        'use_secagg': use_secagg,
        'enable_attack': False,
        'enable_update_clipping': False,
    }
    os.environ['FL_RUN_OVERRIDE'] = _json.dumps(run_cfg)
    client_app = _make_client_app()
    gpu_frac = 0.5 if torch.cuda.is_available() else 0.0
    t0 = time.time()
    run_simulation(
        server_app=_server_app,
        client_app=client_app,
        num_supernodes=num_clients,
        backend_config={'client_resources': {'num_cpus': 2, 'num_gpus': gpu_frac}},
    )
    print(f'  FL done in {time.time()-t0:.0f}s')
    try:
        from federated.server_app import finalize_experiment, save_final_model_from_session
        finalize_experiment()
        if not _pth_ok(model_path):
            save_final_model_from_session(result_dir)
    except Exception as exc:
        print('  finalize_experiment:', exc)
        try:
            from federated.server_app import save_final_model_from_session
            save_final_model_from_session(result_dir)
        except Exception as exc2:
            print('  save_final_model_from_session:', exc2)
    os.environ.pop('FL_RUN_OVERRIDE', None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if not _pth_ok(model_path):
        from federated.server_app import save_final_model_from_session
        save_final_model_from_session(result_dir)

    if not _pth_ok(model_path):
        raise FileNotFoundError(
            f'final_model.pth missing or empty under {result_dir} after training. '
            'Check server logs; ensure finalize_experiment() ran.'
        )
    print(f'  Saved weights -> {model_path} ({model_path.stat().st_size} bytes)')
    return model_path


In [5]:
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import confusion_matrix

from federated.task import get_federated_data
from configs.experiment_config import ModelConfig
from models.resnet_pidl import build_model
from explainability.gradcam import GradCAM, cam_statistics, upsample_cam
from explainability.pm_grid_explainer import (
    gradcam_pm_iou_top25,
    grid_statistics,
    normalize01,
    pm_grid_scores,
    upsample_grid_map,
)
from explainability.plot_utils import overlay_heatmap, savefig_tight, tensor_to_display_rgb


def run_explainability_one_dataset(dataset_name: str, data_root: str, result_dir: Path) -> pd.DataFrame:
    model_path = ensure_final_model(dataset_name, data_root, result_dir)

    fig_dir = result_dir / 'figures'
    for sub in ('gradcam', 'pm_grid', 'combined', 'summary'):
        (fig_dir / sub).mkdir(parents=True, exist_ok=True)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    data = get_federated_data(
        data_root=data_root,
        num_clients=num_clients,
        batch_size=batch_size,
        image_size=image_size,
        random_seed=random_seed,
        save_summary_to=str(result_dir),
    )
    num_classes = data['num_classes']
    class_names = list(data['class_names'])
    test_loader = data['global_test_loader']
    tds = test_loader.dataset
    sub = tds.subset
    base_folder = sub.dataset
    subset_indices = list(sub.indices)

    by_class = {c: [] for c in range(num_classes)}
    for j in range(len(sub)):
        idx = subset_indices[j]
        y = int(base_folder.samples[idx][1])
        by_class[y].append(j)

    rng = random.Random(random_seed)
    picked = []
    for c in range(num_classes):
        pool = by_class[c][:]
        rng.shuffle(pool)
        for j in pool[:samples_per_class]:
            idx = subset_indices[j]
            path = base_folder.samples[idx][0]
            picked.append({
                'ds_index': j,
                'true_class': c,
                'class_name': class_names[c],
                'path': path,
            })

    pd.DataFrame(picked).to_csv(result_dir / 'selected_samples.csv', index=False)

    model_cfg = ModelConfig(pidl_feature_layer=feature_layer)  # type: ignore[arg-type]
    model = build_model(num_classes=num_classes, config=model_cfg)
    model.load_state_dict(torch.load(model_path, map_location='cpu'))
    model.to(device).eval()
    target_mod = getattr(model, gradcam_layer)

    rows, y_true_sel, y_pred_sel = [], [], []
    H = W = image_size

    for i, rec in enumerate(picked):
        image_id = f'{i:04d}'
        x, _ = tds[rec['ds_index']]
        x = x.unsqueeze(0).to(device)

        with GradCAM(model, target_mod) as gcam:
            cam_hw, logits, pred_idx = gcam.compute(x, class_idx=None)

        prob = torch.softmax(logits, dim=-1)[0, pred_idx].item()
        true_idx = rec['true_class']
        ok = pred_idx == true_idx
        y_true_sel.append(true_idx)
        y_pred_sel.append(pred_idx)

        cam_up = upsample_cam(cam_hw, H, W)
        mean_gc, max_gc = cam_statistics(cam_hw)
        model.zero_grad(set_to_none=True)

        with torch.no_grad():
            _, feats = model(x, return_features=True)
            fm_pidl = feats[feature_layer]

        gs = pm_grid_scores(fm_pidl, grid_size=grid_size, k=k_pm, normalize=True)
        gs_n = normalize01(gs)[0]
        pm_up = upsample_grid_map(gs_n.unsqueeze(0), H, W)
        mean_pm, max_pm, top_cell = grid_statistics(gs_n.unsqueeze(0))
        iou_25 = gradcam_pm_iou_top25(cam_up, pm_up)

        rgb = tensor_to_display_rgb(x[0], (H, W))
        o_gc = overlay_heatmap(rgb, cam_up)
        o_pm = overlay_heatmap(rgb, pm_up, cmap='viridis')
        title = (
            f"true={rec['class_name']} | pred={class_names[pred_idx]} | conf={prob:.3f} | ok={ok}"
        )

        fig1, ax = plt.subplots(1, 1, figsize=(4, 4))
        ax.imshow(o_gc)
        ax.set_title('Grad-CAM (classifier grad)\n' + title, fontsize=8)
        ax.axis('off')
        savefig_tight(fig_dir / 'gradcam' / f'{image_id}_gradcam.png')

        fig1, ax = plt.subplots(1, 1, figsize=(4, 4))
        ax.imshow(o_pm)
        ax.set_title('PM grid (physics-informed)\n' + title, fontsize=8)
        ax.axis('off')
        savefig_tight(fig_dir / 'pm_grid' / f'{image_id}_pmgrid.png')

        fig1, axes = plt.subplots(1, 4, figsize=(14, 3.5))
        axes[0].imshow(rgb)
        axes[0].set_title('Input')
        axes[1].imshow(o_gc)
        axes[1].set_title('Grad-CAM')
        axes[2].imshow(o_pm)
        axes[2].set_title('PM (PIDL)')
        axes[3].imshow(np.concatenate([o_gc, o_pm], axis=1))
        axes[3].set_title('GC | PM')
        for ax in axes:
            ax.axis('off')
        fig1.suptitle(title, fontsize=9)
        savefig_tight(fig_dir / 'combined' / f'{image_id}_combined.png')

        rows.append({
            'dataset_name': dataset_name,
            'num_clients': num_clients,
            'image_id': image_id,
            'true_class': rec['class_name'],
            'predicted_class': class_names[pred_idx],
            'confidence': round(prob, 6),
            'correct': ok,
            'mean_gradcam_score': round(mean_gc, 6),
            'max_gradcam_score': round(max_gc, 6),
            'mean_pm_grid_score': round(mean_pm, 6),
            'max_pm_grid_score': round(max_pm, 6),
            'top_pm_grid_index': top_cell,
            'gradcam_pm_overlap_score': round(iou_25, 6),
        })

    summary_df = pd.DataFrame(rows)
    summary_df.to_csv(result_dir / 'explainability_summary.csv', index=False)

    by_cls = summary_df.groupby('true_class').agg(
        gradcam_pm_overlap=('gradcam_pm_overlap_score', 'mean'),
        confidence=('confidence', 'mean'),
    ).reset_index()
    fig2, ax = plt.subplots(figsize=(7, 4))
    ax.bar(by_cls['true_class'], by_cls['gradcam_pm_overlap'], color='steelblue')
    ax.set_ylabel('Mean Grad-CAM / PM IoU (top 25%)')
    ax.set_title(f'{dataset_name}: overlap by class')
    plt.xticks(rotation=30, ha='right')
    savefig_tight(fig_dir / 'summary' / 'bar_overlap_by_class.png')

    fig2, ax = plt.subplots(figsize=(7, 4))
    ax.bar(by_cls['true_class'], by_cls['confidence'], color='darkseagreen')
    ax.set_ylabel('Mean confidence')
    ax.set_title(f'{dataset_name}: confidence by class')
    plt.xticks(rotation=30, ha='right')
    savefig_tight(fig_dir / 'summary' / 'bar_confidence_by_class.png')

    cm = confusion_matrix(y_true_sel, y_pred_sel, labels=list(range(num_classes)))
    fig2, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(num_classes))
    ax.set_yticks(range(num_classes))
    ax.set_xticklabels(class_names, rotation=30, ha='right')
    ax.set_yticklabels(class_names)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    fig2.colorbar(im, ax=ax)
    fig2.suptitle(f'{dataset_name}: confusion (selected samples)')
    savefig_tight(fig_dir / 'summary' / 'confusion_selected.png')

    correct_tbl = summary_df.groupby('correct').size().rename('count').reset_index()
    correct_tbl.to_csv(result_dir / 'explanation_correctness_table.csv', index=False)

    cfg_out = {
        'dataset_name': dataset_name,
        'data_root': data_root,
        'num_clients': num_clients,
        'num_rounds': num_rounds,
        'local_epochs': local_epochs,
        'batch_size': batch_size,
        'image_size': image_size,
        'feature_layer': feature_layer,
        'grid_size': grid_size,
        'lambda_pm': lambda_pm,
        'k_pm': k_pm,
        'gradcam_layer': gradcam_layer,
        'samples_per_class': samples_per_class,
        'random_seed': random_seed,
    }
    (result_dir / 'explainability_config.json').write_text(json.dumps(cfg_out, indent=2))

    print(dataset_name, 'explainability done; rows:', len(summary_df))
    return summary_df


---
## D. Run loop — all datasets


In [6]:
all_parts: list[pd.DataFrame] = []

for dataset_name in DATASETS_TO_RUN:
    print()
    print('========', dataset_name, '========')
    result_dir = EXPLAIN_ROOT / dataset_name / f'{num_clients}_clients'
    data_root = download_data_root(dataset_name)
    print('  image_root:', data_root)
    df = run_explainability_one_dataset(dataset_name, data_root, result_dir)
    all_parts.append(df)

master = pd.concat(all_parts, ignore_index=True)
master_path = EXPLAIN_ROOT / 'explainability_master_summary.csv'
master.to_csv(master_path, index=False)
print()
print('Master summary ->', master_path, f'({len(master)} rows)')
display(master.head(12))



======== brain_tumor_mri ========
Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.
  [brain_tumor_mri] downloaded -> /kaggle/input/brain-tumor-mri-dataset
[find_image_root] Found (named training split): 'Training'
  Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
  image_root: /kaggle/input/brain-tumor-mri-dataset/Training
  Running Flower FL (required for this notebook — no reuse of notebook 01 weights)...



            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
DEBUG:flwr:backend_config: {'client_resources': {'num_cpus': 2, 'num_gpus': 0.5}}
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
DEBUG:flwr:Asyncio event loop al

[Server] Device: cuda  |  Log dir: /content/medical_fl_pidl/results_explainability/brain_tumor_mri/3_clients
[task] Building federated data for 3 clients from: /kaggle/input/brain-tumor-mri-dataset/Training
[build_federated_dataloaders] Loading ImageFolder from '/kaggle/input/brain-tumor-mri-dataset/Training' …
  → 5,600 images  |  4 classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
  → Grayscale source detected. Images will be converted to 3-channel RGB.
  → Split: 4,480 train  |  1,120 test  (stratified, seed=42)
  → Partitioning (iid): client 0: 1,496  |  client 1: 1,492  |  client 2: 1,492
[dataset_utils] Summary saved → /content/medical_fl_pidl/results_explainability/brain_tumor_mri/3_clients/dataset_summary.json
[build_federated_dataloaders] Done. 3 client loaders + 1 test loader ready.

[Server] Dataset  : brain_tumor_mri  (4 classes: ['glioma', 'meningioma', 'notumor', 'pituitary'])
[Server] Test set : 1120 samples


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


...

Round   5 | Train Acc: 87.37%  Loss: 0.3528  PIDL: 0.019783 | Client Val Acc: 91.28%  Loss: 0.2415 |  Server Acc: 91.28% | Elapsed: 450.6s
  FL done in 2306s

  EXPERIMENT SUMMARY
  Dataset:  covid   Clients: 3   Rounds: 6
  Best Accuracy             0.9128  (round 5)
  Final Accuracy            0.9128
  Best Balanced Acc         0.9086  (round 3)
  Final Balanced Acc        0.9056
  Best Macro F1             0.9196  (round 5)
  Final Macro F1            0.9196
  Best ROC-AUC              0.9867  (round 5)
  Best PR-AUC               0.9727  (round 5)
  Final ECE                 0.0092
  Train time (total)        0.0000
  Infer time (total)        155.6400


[ExperimentLogger] All outputs written to: /content/medical_fl_pidl/results_explainability/covid/3_clients
  fl_rounds.csv        (6 rounds)
  per_class_metrics.csv
  fl_eval.json
  fl_summary.json


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[finalize_experiment] Final model saved -> /content/medical_fl_pidl/results_explainability/covid/3_clients/final_model.pth
  Saved weights -> /content/medical_fl_pidl/results_explainability/covid/3_clients/final_model.pth (44792395 bytes)
covid explainability done; rows: 20

Master summary -> /content/medical_fl_pidl/results_explainability/explainability_master_summary.csv (50 rows)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,dataset_name,num_clients,image_id,true_class,predicted_class,confidence,correct,mean_gradcam_score,max_gradcam_score,mean_pm_grid_score,max_pm_grid_score,top_pm_grid_index,gradcam_pm_overlap_score
0,brain_tumor_mri,3,0000,glioma,glioma,0.958121,True,0.296245,1.0,0.428770,1.0,14,0.124669
1,brain_tumor_mri,3,0001,glioma,glioma,0.998897,True,0.314052,1.0,0.330271,1.0,5,0.469789
2,brain_tumor_mri,3,0002,glioma,glioma,0.994045,True,0.425186,1.0,0.424888,1.0,14,0.362126
3,brain_tumor_mri,3,0003,glioma,glioma,0.985672,True,0.409389,1.0,0.334980,1.0,9,0.209877
4,brain_tumor_mri,3,0004,glioma,glioma,0.999242,True,0.362384,1.0,0.344797,1.0,10,0.792128
5,brain_tumor_mri,3,0005,meningioma,pituitary,0.613129,False,0.318100,1.0,0.448257,1.0,14,0.078080
6,brain_tumor_mri,3,0006,meningioma,meningioma,0.984284,True,0.256075,1.0,0.433646,1.0,6,0.126892
7,brain_tumor_mri,3,0007,meningioma,meningioma,0.983320,True,0.306552,1.0,0.470926,1.0,11,0.066485
8,brain_tumor_mri,3,0008,meningioma,meningioma,0.997999,True,0.236523,1.0,0.434840,1.0,14,0.101414
9,brain_tumor_mri,3,0009,meningioma,meningioma,0.823316,True,0.278974,1.0,0.406983,1.0,14,0.138656


---
## E. Download entire `results_explainability/` (includes all `.pth` files)


In [7]:
import subprocess

# Verify every dataset has a non-empty checkpoint before zipping
for ds in DATASETS_TO_RUN:
    exp_pth = EXPLAIN_ROOT / ds / f'{num_clients}_clients' / 'final_model.pth'
    if not exp_pth.is_file() or exp_pth.stat().st_size <= 1024:
        raise FileNotFoundError(f'Checkpoint missing or empty — will not zip: {exp_pth}')

for p in sorted(EXPLAIN_ROOT.rglob('final_model.pth')):
    print(p, p.stat().st_size, 'bytes')

zip_path = '/content/explainability_results_full.zip'
subprocess.run(['zip', '-r', zip_path, 'results_explainability'], cwd=str(PROJECT_ROOT), check=True)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('ZIP at', zip_path)


/content/medical_fl_pidl/results_explainability/brain_tumor_mri/3_clients/final_model.pth 44792395 bytes
/content/medical_fl_pidl/results_explainability/colon_cancer_or_pathology/3_clients/final_model.pth 44788299 bytes
/content/medical_fl_pidl/results_explainability/covid/3_clients/final_model.pth 44792395 bytes


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>